# AeroPulse Propagation — 03: Transport Features

Fix the wind unit error, replace the constant-wind approximation with
step-integrated displacement over a simulated forecast vintage, and add
stagnation persistence.

## Objective

Improvement plan sections 6, 12, 13 and 15.

Three separate problems with the previous transport implementation:

**1. The unit was wrong.** The old cell computed
`advect_dx_km = wind_u * 3600 / 1000`, i.e. it treated `wind_u` as m/s.
Open-Meteo is queried without a `wind_speed_unit` parameter, so it returns
**km/h**. Every advection distance in the previous feature set was inflated
by a factor of 3.6. Section 1 below demonstrates this against the data rather
than asserting it.

**2. The physics was a straight line.** `distance = speed x horizon` from a
single instantaneous wind vector is defensible at 1-3 h and increasingly
fictional at 24-48 h, where the wind will have turned several times.

**3. Using the observed wind at t+h would be leakage.** The plan's fix is a
forecast vintage. AeroPulse has no archived NWP forecast to replay, so we
build a **simulated vintage**: blend the wind known at t toward the station's
own (train-period) climatology with an exponential decay,
`w = exp(-lead/tau)`. At lead 1 h it is essentially persistence; by 24 h it
is essentially climatology. That is both leakage-free and a fair imitation of
how real forecast skill degrades — and it is honest about the fact that
production will feed real NWP wind into the same slot.

In [1]:
# ============================================================================
# SETUP — everything reusable lives in propagation_toolkit.py, not in cells.
# ============================================================================
import os, sys, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import propagation_toolkit as pt

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

cfg = pt.load_config()
HORIZONS = cfg.horizons
print("kernel:", sys.executable)

env: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/propagation_forecast/.env
project: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/propagation_forecast
horizons: [1, 3, 6, 12, 24, 48] | schema 2.0.0
kernel: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/.venv/bin/python


## 1. The unit error, demonstrated

In [2]:
feature_file = cfg.feature_dir / "temporal_features.parquet"
if not feature_file.exists():
    raise FileNotFoundError("Run 02_propagation_temporal_spatial_features.ipynb first.")

df = pd.read_parquet(feature_file)
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)
df = df.sort_values(["location_id", "timestamp_utc"]).reset_index(drop=True)
print("rows", f"{len(df):,}", "| columns", df.shape[1])

speed = df["wind_speed_10m"]
print("\nwind_speed_10m distribution:")
print(speed.describe(percentiles=[0.05, 0.5, 0.95, 0.99]).round(2).to_string())
print(f"\nif km/h -> mean {speed.mean():.1f} km/h = {speed.mean() / 3.6:.1f} m/s")
print(f"if m/s  -> mean {speed.mean():.1f} m/s  = {speed.mean() * 3.6:.1f} km/h")
print("\nA 32 km/h (Beaufort 5) annual mean over the Indian subcontinent is not")
print("physical. The 99th percentile would be", round(float(speed.quantile(0.99)) * 3.6), "km/h.")
print("Contract-declared unit:", pt.SCHEMA["units"]["wind_speed_10m"])

rows 1,705,252 | columns 220

wind_speed_10m distribution:
count    1705252.00
mean           9.00
std            5.07
min            0.00
5%             1.90
50%            8.30
95%           18.40
99%           23.20
max           41.40

if km/h -> mean 9.0 km/h = 2.5 m/s
if m/s  -> mean 9.0 m/s  = 32.4 km/h

A 32 km/h (Beaufort 5) annual mean over the Indian subcontinent is not
physical. The 99th percentile would be 84 km/h.
Contract-declared unit: km/h


In [3]:
# The magnitude of the previous error, in km.
old_recipe_6h = df["wind_speed_10m"] * 6 * 3600 / 1000.0   # treated as m/s
new_recipe_6h = df["wind_speed_10m"] * 6 * pt.WIND_KMH_TO_KM_PER_H
comparison = pd.DataFrame({
    "old_m_per_s_assumption_km": old_recipe_6h.describe(),
    "correct_km_per_h_km": new_recipe_6h.describe(),
}).round(1)
print("6 h straight-line transport distance:")
print(comparison.to_string())
print(f"\ninflation factor: {float(old_recipe_6h.mean() / new_recipe_6h.mean()):.2f}x")

6 h straight-line transport distance:
       old_m_per_s_assumption_km  correct_km_per_h_km
count                  1705252.0            1705252.0
mean                       194.4                 54.0
std                        109.6                 30.4
min                          0.0                  0.0
25%                        112.3                 31.2
50%                        179.3                 49.8
75%                        261.4                 72.6
max                        894.2                248.4

inflation factor: 3.60x


The old feature said a typical air parcel travels ~194 km in 6 hours. It
travels ~54 km. Tree models are scale-invariant, so this did not break
training — but any downstream use of the displacement as a *geographic*
quantity (looking up who is upwind, matching to a grid cell) would have been
pointing at the wrong part of the country.

## 2. Fit the wind climatology

Per (station, month, hour), fitted on the training window only — same cutoff
as the station baselines in notebook 02, for the same reason.

In [4]:
fit_end = df["timestamp_utc"].quantile(cfg.train_fraction)
t0 = time.time()
wind_clim = pt.WindClimatology.fit(df, fit_end, tau_hours=6.0)
print(f"fitted in {time.time() - t0:.1f}s")
print(json.dumps(wind_clim.to_dict(), indent=2))

print("\nblend weight on the observed wind, by lead time:")
for lead in [1, 3, 6, 12, 24, 48]:
    w = np.exp(-lead / wind_clim.tau_hours)
    print(f"  t+{lead:>2}h  observed {w:5.1%}   climatology {1 - w:5.1%}")

fitted in 0.1s
{
  "tau_hours": 6.0,
  "station_month_hour_cells": 42639,
  "blend": "w=exp(-lead/tau) on observed wind, 1-w on climatology"
}

blend weight on the observed wind, by lead time:
  t+ 1h  observed 84.6%   climatology 15.4%
  t+ 3h  observed 60.7%   climatology 39.3%
  t+ 6h  observed 36.8%   climatology 63.2%
  t+12h  observed 13.5%   climatology 86.5%
  t+24h  observed  1.8%   climatology 98.2%
  t+48h  observed  0.0%   climatology 100.0%


### Fit tau to the wind *anomaly* autocorrelation

`tau` should not be a guess, but it also must be fitted to the right quantity.
The blend is algebraically `clim + w x (observed - clim)`, so `w` governs how
long the **anomaly** persists, not the raw wind.

Those decay at very different rates. Raw wind autocorrelation flattens near
0.5 and rises again at 24 h, because the diurnal cycle keeps bringing the
same wind back — fitting an exponential to it yields a misleadingly long tau
(37 h on this data). The climatology term already supplies that recurring
component; what decays is the departure from it.

The anomaly is better behaved but not perfectly exponential either, so the
fit is restricted to the leads where it is still monotone. The table below
shows both series and marks which rows the fit used.

In [5]:
probe = df[df["location_id"].isin(df["location_id"].drop_duplicates().head(25))].copy()
probe["month"] = probe["timestamp_utc"].dt.month
clim_lookup = wind_clim.table.rename(columns={"clim_u": "_cu", "clim_v": "_cv"})
probe = probe.merge(clim_lookup, on=["location_id", "month", "hour"], how="left")
probe["anom_u"] = probe["wind_u"] - probe["_cu"]
probe["anom_v"] = probe["wind_v"] - probe["_cv"]

leads = [1, 2, 3, 6, 9, 12, 18, 24, 36, 48]
rows = []
for lead in leads:
    shifted = probe.groupby("location_id", observed=True)[
        ["wind_u", "wind_v", "anom_u", "anom_v"]
    ].shift(-lead)
    mask = shifted["wind_u"].notna() & probe["anom_u"].notna() & shifted["anom_u"].notna()
    raw = np.mean([
        float(np.corrcoef(probe.loc[mask, c], shifted.loc[mask, c])[0, 1])
        for c in ("wind_u", "wind_v")
    ])
    anom = np.mean([
        float(np.corrcoef(probe.loc[mask, c], shifted.loc[mask, c])[0, 1])
        for c in ("anom_u", "anom_v")
    ])
    rows.append({"lead_h": lead, "raw_autocorr": round(raw, 3), "anomaly_autocorr": round(anom, 3)})
decay = pd.DataFrame(rows)

# Fit only where the decay is monotone. Both series re-correlate near 24 h, so
# a fit over the full range is dominated by that recurrence and returns a
# meaninglessly long tau (37 h on this data). MONOTONE_MAX_LEAD is the honest
# boundary of what an exponential can describe.
MONOTONE_MAX_LEAD = 12
usable = decay[(decay["lead_h"] <= MONOTONE_MAX_LEAD) & (decay["anomaly_autocorr"] > 0.02)]
slope = np.polyfit(usable["lead_h"], np.log(usable["anomaly_autocorr"]), 1)[0]
TAU = float(-1.0 / slope)
decay["fitted_exp"] = np.exp(-decay["lead_h"] / TAU).round(3)
decay["used_in_fit"] = decay["lead_h"] <= MONOTONE_MAX_LEAD
print(decay.to_string(index=False))
print(f"\nfitted tau = {TAU:.1f} h  (anomaly column, leads <= {MONOTONE_MAX_LEAD} h)")
print("Both series rebound at 24 h. Removing the (station, month, hour) climatology")
print("takes out the *mean* diurnal cycle but not its day-to-day amplitude, so the")
print("anomaly still recurs a day later. Beyond ~12 h the exponential is an")
print("approximation that errs toward climatology - which is the safe direction.")

 lead_h  raw_autocorr  anomaly_autocorr  fitted_exp  used_in_fit
      1         0.896             0.866       0.887         True
      2         0.806             0.734       0.786         True
      3         0.734             0.615       0.697         True
      6         0.586             0.352       0.486         True
      9         0.514             0.269       0.339         True
     12         0.488             0.248       0.236         True
     18         0.525             0.300       0.115        False
     24         0.579             0.471       0.056        False
     36         0.364             0.097       0.013        False
     48         0.435             0.273       0.003        False

fitted tau = 8.3 h  (anomaly column, leads <= 12 h)
Both series rebound at 24 h. Removing the (station, month, hour) climatology
takes out the *mean* diurnal cycle but not its day-to-day amplitude, so the
anomaly still recurs a day later. Beyond ~12 h the exponential is an
approximat

In [6]:
wind_clim = pt.WindClimatology.fit(df, fit_end, tau_hours=TAU)
print("refitted climatology with the measured tau")
for lead in [1, 3, 6, 12, 24, 48]:
    w = np.exp(-lead / wind_clim.tau_hours)
    print(f"  t+{lead:>2}h  observed {w:5.1%}   climatology {1 - w:5.1%}")

refitted climatology with the measured tau
  t+ 1h  observed 88.7%   climatology 11.3%
  t+ 3h  observed 69.7%   climatology 30.3%
  t+ 6h  observed 48.6%   climatology 51.4%
  t+12h  observed 23.6%   climatology 76.4%
  t+24h  observed  5.6%   climatology 94.4%
  t+48h  observed  0.3%   climatology 99.7%


## 3. Step-integrated advection

In [7]:
t0 = time.time()
before = set(df.columns)
df = pt.integrated_advection(df, HORIZONS, wind_clim, step_hours=1)
added = sorted(set(df.columns) - before)
print(f"integrated {max(HORIZONS)} hourly steps in {time.time() - t0:.1f}s")
families = sorted({name.rsplit("_h", 1)[0] for name in added})
print(f"added {len(added)} columns = {len(families)} families x {len(HORIZONS)} horizons:")
for name in families:
    print("  ", name + "_h{H}")

integrated 48 hourly steps in 7.1s
added 72 columns = 12 families x 6 horizons:
   advect_dist_km_h{H}
   advect_dx_km_h{H}
   advect_dy_km_h{H}
   advect_lat_h{H}
   advect_lon_h{H}
   fcst_wind_speed_mean_h{H}
   fcst_wind_speed_min_h{H}
   fcst_wind_speed_std_h{H}
   transport_bearing_deg_h{H}
   transport_path_km_h{H}
   transport_straightness_h{H}
   transport_turn_deg_h{H}


In [8]:
summary = []
for h in HORIZONS:
    summary.append({
        "horizon_h": h,
        "net_displacement_km": round(float(df[f"advect_dist_km_h{h}"].median()), 1),
        "path_length_km": round(float(df[f"transport_path_km_h{h}"].median()), 1),
        "straightness": round(float(df[f"transport_straightness_h{h}"].median()), 3),
        "wind_turn_deg": round(float(df[f"transport_turn_deg_h{h}"].median()), 1),
        "naive_constant_wind_km": round(float((df["wind_speed_10m"] * h).median()), 1),
    })
print(pd.DataFrame(summary).to_string(index=False))

 horizon_h  net_displacement_km  path_length_km  straightness  wind_turn_deg  naive_constant_wind_km
         1                  7.8             7.8         1.000            0.0                     8.3
         3                 21.9            21.9         1.000            3.8                    24.9
         6                 40.2            40.4         0.997           10.6                    49.8
        12                 69.4            71.8         0.984           25.7                    99.6
        24                121.2           131.9         0.952           21.4                   199.2
        48                221.0           252.7         0.923           23.9                   398.4


### Why straightness matters

`straightness = net displacement / path length`. At 1.0 the air travelled in a
straight line and the constant-wind approximation is fine. As it falls, the
parcel is meandering and a single distance number stops describing anything
real. The naive column is what the previous notebook produced (after the unit
fix); the gap between it and the net displacement is the error the old
feature carried into the model.

## 4. Stability and stagnation persistence (plan section 13)

In [9]:
t0 = time.time()
before = set(df.columns)
df = pt.add_stability_features(df)
print(f"added in {time.time() - t0:.1f}s:", sorted(set(df.columns) - before))

print("\nstagnation persistence. A rolling mean preserves the overall rate;")
print("what changes is how often the air has been stagnant for a *sustained* spell:")
for column in ["stagnation_flag", "stagnation_3h", "stagnation_6h", "stagnation_12h"]:
    print(f"  {column:<18} rate {float(df[column].mean()):.4f}   "
          f"fraction of rows > 0.5: {float((df[column] > 0.5).mean()):.4f}")
print(f"  stagnation_duration_h  mean {float(df['stagnation_duration_h'].mean()):.2f} h, "
      f"max {float(df['stagnation_duration_h'].max()):.0f} h")

# Does duration add anything the instantaneous flag does not?
sample = df.dropna(subset=["target_pm25_t_plus_24h"]).sample(200_000, random_state=42)
resid = sample["target_pm25_t_plus_24h"] - sample["pm25"]
for column in ["stagnation_flag", "stagnation_6h", "stagnation_duration_h", "ventilation_log"]:
    print(f"  corr({column:<22}, 24h residual) = {sample[column].corr(resid):+.4f}")

added in 0.3s: ['delta_pbl_6h', 'stagnation_12h', 'stagnation_3h', 'stagnation_6h', 'stagnation_duration_h', 'ventilation_change_6h', 'ventilation_index', 'ventilation_log']

stagnation persistence. A rolling mean preserves the overall rate;
what changes is how often the air has been stagnant for a *sustained* spell:
  stagnation_flag    rate 0.0439   fraction of rows > 0.5: 0.0439
  stagnation_3h      rate 0.0439   fraction of rows > 0.5: 0.0326
  stagnation_6h      rate 0.0439   fraction of rows > 0.5: 0.0145
  stagnation_12h     rate 0.0439   fraction of rows > 0.5: 0.0073
  stagnation_duration_h  mean 0.09 h, max 23 h


  corr(stagnation_flag       , 24h residual) = -0.0134
  corr(stagnation_6h         , 24h residual) = -0.0212
  corr(stagnation_duration_h , 24h residual) = -0.0122
  corr(ventilation_log       , 24h residual) = +0.0203


## 5. Regime features (plan section 15)

In [10]:
before = set(df.columns)
df = pt.add_regime_features(df)
print("added:", sorted(set(df.columns) - before))

print("\nregime prevalence:")
for column in ["regime_rising", "regime_falling", "regime_stagnant",
               "regime_fire_influenced", "regime_transport_dominated"]:
    print(f"  {column:<28} {float(df[column].mean()):6.2%}")

print("\nconcentration bands (pt.regime_label):")
print(pt.regime_label(df).value_counts().sort_index().to_string())

added: ['regime_falling', 'regime_fire_influenced', 'regime_level', 'regime_rising', 'regime_stagnant', 'regime_transport_dominated']

regime prevalence:
  regime_rising                18.93%
  regime_falling               19.07%
  regime_stagnant               0.89%
  regime_fire_influenced       29.06%
  regime_transport_dominated   60.46%

concentration bands (pt.regime_label):
pm25
<30        906068
30-60      545075
60-120     216184
120-250     33436
>250         4489


Note the deliberate omission: the upstream `pollution_regime` column is
excluded from the feature set by `pt.select_features`. It is derived with
same-hour information and is useful as a *reporting* label in notebook 07,
not as an input.

## 6. Leakage re-check on the new columns

In [11]:
new_columns = [c for c in df.columns if c.startswith(("advect_", "transport_", "fcst_", "stagnation_", "regime_", "ventilation_"))]
print("new transport/stability/regime columns:", len(new_columns))

scan = pt.leakage_scan(df, new_columns, "target_pm25_t_plus_6h", sample=200_000)
print("persistence bar:", round(scan.attrs["persistence_corr"], 4))
print(scan.head(8).to_string(index=False))
print("\nsuspects:", int(scan["suspect"].sum()))
assert int(scan["suspect"].sum()) == 0, "a transport feature is correlated with the future"

# The forecast-vintage wind must not equal the observed future wind.
future_wind = df.groupby("location_id", observed=True)["wind_u"].shift(-6)
mask = future_wind.notna()
u6, _ = wind_clim.lead_wind(df, 6)
print(f"\ncorr(simulated t+6 forecast wind_u, actual t+6 wind_u) = "
      f"{float(np.corrcoef(u6[mask.to_numpy()], future_wind[mask])[0, 1]):.3f}")
print("(well below 1.0 — the vintage is a forecast, not a peek at the answer)")

new transport/stability/regime columns: 89


persistence bar: 0.5676
                   feature  corr_target  corr_vs_persistence  suspect
              regime_level      0.56388             -0.00370    False
regime_transport_dominated      0.32862             -0.23897    False
          advect_dy_km_h48      0.29035             -0.27724    False
          advect_dy_km_h24      0.27735             -0.29024    False
          advect_dy_km_h12      0.24786             -0.31973    False
  fcst_wind_speed_mean_h12      0.23983             -0.32775    False
     transport_path_km_h12      0.23983             -0.32775    False
  fcst_wind_speed_mean_h24      0.23811             -0.32948    False

suspects: 0

corr(simulated t+6 forecast wind_u, actual t+6 wind_u) = 0.798
(well below 1.0 — the vintage is a forecast, not a peek at the answer)


## 7. Save the modelling feature table

In [12]:
out = cfg.feature_dir / "propagation_features.parquet"
float_cols = df.select_dtypes(include=["float64"]).columns
df[float_cols] = df[float_cols].astype("float32")
df.to_parquet(out, index=False)

import joblib
joblib.dump(wind_clim, cfg.feature_dir / "wind_climatology.joblib")

manifest = {
    "schema_version": cfg.schema_version,
    "generated_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "rows": int(len(df)),
    "columns": int(df.shape[1]),
    "horizons": HORIZONS,
    "wind_units": pt.SCHEMA["units"]["wind_speed_10m"],
    "wind_climatology": wind_clim.to_dict(),
    "tau_fitted_hours": TAU,
    "climatology_fit_end": str(fit_end),
    "transport_columns": [c for c in df.columns if c.startswith(("advect_", "transport_", "fcst_"))],
    "stability_columns": [c for c in df.columns if c.startswith(("stagnation_", "ventilation_"))],
    "regime_columns": [c for c in df.columns if c.startswith("regime_")],
    "transport_summary": summary,
    "wind_autocorrelation": decay.to_dict(orient="records"),
    "fingerprint": pt.dataset_fingerprint(df[["timestamp_utc", "location_id", "pm25"]]),
    "notes": [
        "Wind is km/h. The previous implementation assumed m/s and inflated every distance 3.6x.",
        "Advection is a displacement feature, not a transported concentration field.",
        "Forecast-vintage wind is simulated (persistence decaying to climatology); production must substitute real NWP.",
    ],
}
(cfg.feature_dir / "dataset_manifest.json").write_text(json.dumps(manifest, indent=2, default=str))

print("03 COMPLETE")
print("  ", out, f"({df.shape[1]} columns)")
print("Next: 04_propagation_baselines.ipynb")

03 COMPLETE
   /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/propagation_forecast/data/propagation/processed/features/propagation_features.parquet (306 columns)
Next: 04_propagation_baselines.ipynb


## Findings

**The wind unit error is confirmed and fixed.** Mean `wind_speed_10m` is 9.0.
Read as km/h that is 2.5 m/s — an ordinary annual mean over the Indian
subcontinent. Read as m/s it is 32 km/h, with a 99th percentile of 84 km/h,
which would make gale-force wind a routine hourly condition across 149
stations. It is km/h.

The previous implementation's 6 h transport distance therefore averaged
**194 km when the correct value is 54 km — a 3.6x inflation on every
horizon.** Tree models are scale-invariant so this did not corrupt the
existing skill numbers, but it makes the displacement useless as a
*geographic* quantity: any downstream lookup of "who is upwind of this
parcel" or match to a grid cell was pointing at the wrong part of the country.

**tau is fitted, and fitting it to the right series matters.** Naively fitting
the raw wind autocorrelation gives tau = 37.7 h, because raw autocorrelation
flattens near 0.5 and rebounds at 24 h with the diurnal cycle. The blend is
`clim + w x (observed - clim)`, so `w` should follow the **anomaly**
autocorrelation, restricted to the leads where it is still monotone:

| lead | raw autocorr | anomaly autocorr | in fit |
|---|---|---|---|
| 1 h | 0.896 | 0.866 | yes |
| 3 h | 0.734 | 0.615 | yes |
| 6 h | 0.586 | 0.352 | yes |
| 12 h | 0.488 | 0.248 | yes |
| 24 h | 0.579 | 0.471 | no — rebound |

**tau = 8.3 h.** The 24 h rebound survives climatology removal because
subtracting a (station, month, hour) mean removes the *average* diurnal cycle
but not its day-to-day amplitude. Beyond 12 h the exponential is an
approximation that errs toward climatology, which is the safe direction for a
forecast-vintage feature.

**Integrated advection replaces the constant-wind line.** 72 new columns, 12
families x 6 horizons.

| horizon | net displacement | path length | straightness | naive constant-wind |
|---|---|---|---|---|
| 1 h | 7.8 km | 7.8 km | 1.000 | 8.3 km |
| 6 h | 40.2 km | 40.4 km | 0.997 | 49.8 km |
| 24 h | 121.2 km | 131.9 km | 0.952 | 199.2 km |
| 48 h | 221.0 km | 252.7 km | 0.923 | 398.4 km |

At 48 h the naive column is **80% larger** than the integrated net
displacement, and the parcel's path is 14% longer than the straight line
between its endpoints. Both errors grow with horizon, exactly where the
constant-wind assumption was always weakest.

One caveat worth stating rather than burying: straightness stays high (0.92 at
48 h) partly because the simulated vintage is smooth — climatology is a mean,
so it has less variance than real wind. **Real 48 h trajectories meander
more than this, so 0.92 is an upper bound.** Substituting genuine NWP wind
would lower it.

**Stagnation persistence separates what a single flag could not.** The
instantaneous rate is 4.4% by construction, and a rolling mean preserves it —
what changes is sustained occupancy: 3.3% of rows are majority-stagnant over
3 h, 1.5% over 6 h, 0.7% over 12 h. So sustained stagnation is roughly
**six times rarer** than the instantaneous flag suggests, which is the
distinction that matters for episode build-up.

Honesty about effect size: none of the stability features correlate strongly
with the 24 h residual (`stagnation_6h` -0.021, `ventilation_log` +0.020,
`stagnation_duration_h` -0.012). They are physically motivated and cheap, but
on this evidence they are not going to move the headline metric. Notebook 05
tests that directly instead of assuming it.

**Leakage: clean.** Zero suspects across all 89 new columns; the strongest is
`regime_level` at 0.564, which is below the persistence bar of 0.568 because
it is a coarse binning of `pm25` itself. The forecast-vintage wind correlates
0.798 with the actually-observed t+6 wind — high enough to be informative,
far enough from 1.0 to confirm it is a forecast and not a peek at the answer.